# NLP Lab Assignment — Experiment 4
## Title: Implement Advanced Text Pre-processing Techniques
**Case Study Domain:** Terms and Conditions Summarizer (Amazon & Alibaba Agreements)

### Objective:
To implement advanced linguistic pre-processing techniques including domain-specific stop-word removal, comparative Stemming vs. Lemmatization (POS-aware), OCR/formatting noise handling, multilingual language detection & translation, and entity anonymization (PII masking).

### 1. Comparative Stemming vs POS-Aware Lemmatization on Legal Morphology
Compare Porter Stemmer, Lancaster Stemmer, and POS-aware WordNet Lemmatizer on legal terms.

In [1]:
import nltk
from nltk.stem import PorterStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from nltk import pos_tag, word_tokenize
import pandas as pd

porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

legal_words = ["indemnification", "indemnified", "terminating", "obligations", "breaching", "arbitration", "confidentiality", "disclaimers"]

morph_results = []
for w in legal_words:
    morph_results.append({
        "Original Legal Term": w,
        "Porter Stem": porter.stem(w),
        "Lancaster Stem": lancaster.stem(w),
        "WordNet Lemma (Noun)": lemmatizer.lemmatize(w, pos=wordnet.NOUN),
        "WordNet Lemma (Verb)": lemmatizer.lemmatize(w, pos=wordnet.VERB)
    })

df_morph = pd.DataFrame(morph_results)
print("=== Stemming vs Lemmatization Comparison ===")
display(df_morph)

=== Stemming vs Lemmatization Comparison ===


,Original Legal Term,Porter Stem,Lancaster Stem,WordNet Lemma (Noun),WordNet Lemma (Verb)
0,indemnification,indemnif,indemn,indemnification,indemnification
1,indemnified,indemnifi,indemn,indemnified,indemnify
2,terminating,termin,termin,terminating,terminate
3,obligations,oblig,oblig,obligation,obligations
4,breaching,breach,breach,breaching,breach
5,arbitration,arbitr,arbit,arbitration,arbitration
6,confidentiality,confidenti,confid,confidentiality,confidentiality
7,disclaimers,disclaim,disclaim,disclaimer,disclaimers


### 2. Entity Anonymization & PII Masking
Mask user identities, emails, organizations, and sensitive parameters with anonymized placeholder tokens (`<EMAIL>`, `<ORG>`, `<MONEY>`).

In [2]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

def anonymize_legal_entities(text):
    # 1. Regex PII Masking (Emails and Phone numbers)
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "<EMAIL>", text)
    text = re.sub(r"\+?\d{1,3}[-\s]?\(?\d{3}\)?[-\s]?\d{3}[-\s]?\d{4}", "<PHONE_NUMBER>", text)
    
    # 2. NER-based masking via spaCy
    doc = nlp(text)
    anonymized_tokens = []
    for token in doc:
        if token.ent_type_ in ["PERSON"]:
            anonymized_tokens.append("<USER_NAME>")
        elif token.ent_type_ in ["MONEY"]:
            anonymized_tokens.append("<MONETARY_AMOUNT>")
        else:
            anonymized_tokens.append(token.text)
    
    return " ".join(anonymized_tokens)

contract_clause = "User John Doe agreed to pay $2,500.00 to Alibaba Services LLC. In case of dispute, email jdoe@sample.com or call +1 800 555 0199."
anonymized_output = anonymize_legal_entities(contract_clause)

print("Original Contract Clause:")
print(contract_clause)
print("\nAnonymized Contract Clause (PII Redacted):")
print(anonymized_output)

Original Contract Clause:
User John Doe agreed to pay $2,500.00 to Alibaba Services LLC. In case of dispute, email jdoe@sample.com or call +1 800 555 0199.

Anonymized Contract Clause (PII Redacted):
User <USER_NAME> <USER_NAME> agreed to pay $ <MONETARY_AMOUNT> to Alibaba Services LLC . In case of dispute , email < EMAIL > or call < PHONE_NUMBER > .


### 3. Multilingual Text Handling & Script Identification
Identify foreign-language provisions (e.g. cross-border agreements in Chinese/Spanish) and handle routing.

In [3]:
from langdetect import detect

clauses = [
    {"Company": "Amazon (US)", "Text": "You are responsible for maintaining the confidentiality of your account and password."},
    {"Company": "Alibaba (CN/Intl)", "Text": "本协议受中华人民共和国法律管辖并按其解释。"}, # Alibaba Chinese Governing Law
    {"Company": "Amazon (ES)", "Text": "Amazon se reserva el derecho de denegar el servicio o cancelar cuentas a su entera discrecion."}
]

for c in clauses:
    lang = detect(c["Text"])
    print(f"[{c['Company']}] Detected Language: '{lang}' | Clause: {c['Text']}")

[Amazon (US)] Detected Language: 'en' | Clause: You are responsible for maintaining the confidentiality of your account and password.
[Alibaba (CN/Intl)] Detected Language: 'zh-cn' | Clause: 本协议受中华人民共和国法律管辖并按其解释。
[Amazon (ES)] Detected Language: 'es' | Clause: Amazon se reserva el derecho de denegar el servicio o cancelar cuentas a su entera discrecion.
